### Workbook for exploring datasets from Uganda Ministry of Agriculture, Animal Industry and Fisheries
Week of November 24, 2025
<br>
Author: Adele Birkenes

In [1]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, LineString, Polygon
import os
import matplotlib.pyplot as plt
import numpy as np

In [2]:
unsynced_data_path = "../../unsynced-data/Uganda_ag_ministry_shapefiles"

In [5]:
# Print names of all folders in the unsynced data path in alphabetical order
for folder_name in sorted(os.listdir(unsynced_data_path)):
    if os.path.isdir(os.path.join(unsynced_data_path, folder_name)):
        print(folder_name)

2021 subcounties
Alebtong SC
Apac SC
Cindy Parishes
District 2019
District 2020
District 2021
District_City
Districts_2020_146_with Cities
Dokolo Updated SC
Kigezi
Lango
Lango City
Lango Parish
Lira City Parishes
Lira DSC
Oyam Parishes
Oyam Updated SC
Subcounties_2019
UgaGIS_data
Uganda Education
Uganda Rivers
Uganda Roads
Uganda Shapefile
Uganda Water
Uganda health_facilities
Uganda villages
Uganda-Subcounties-2021_Duncan
Uganda_parishes


In [10]:
# For a given folder, list all shapefiles, even if they are nested in subfolders
# Only print the names of the shapefiles and their subfolders, not the full file paths
def list_shapefiles(folder_path):
    folder = os.path.join(unsynced_data_path, folder_path)
    shapefiles = []
    for root, dirs, files in os.walk(folder):
        for f in files:
            if f.endswith('.shp'):
                shapefiles.append(os.path.relpath(os.path.join(root, f), folder))
    return shapefiles

In [11]:
list_shapefiles("2021 subcounties")

['Lango Updated Subcounties/Lango Updated.shp',
 'Lira District/Lira District.shp',
 'Kole/Kole Subcounties.shp',
 'Amolatar/Amolatar.shp',
 'Oyam/Oyam Subcounties.shp',
 'Otuke/Otuke.shp',
 'Apac/Apac Subcounties.shp',
 'Kwania/Kwania Subcouties.shp',
 'Dokolo/Dokolo.shp',
 'Lira City/Lira City.shp',
 'Alebtong/Alebtong.shp']

In [22]:
# For a given shapefile, read it in as a geodataframe and print the attribute table
def read_shapefile(shapefile_path):
    shapefile = os.path.join(unsynced_data_path, shapefile_path)
    try:
        return gpd.read_file(shapefile)
    except Exception as e:
        print(f"Error reading {shapefile}: {e}")
        return None

In [23]:
read_shapefile("2021 subcounties/Lango Updated Subcounties/Lango Updated.shp")

,OBJECTID_1,District,Subcounty,Sub_County,geometry
0,15,OYAM,ABER,ABER,"POLYGON ((439806.031 10248637.263, 439856.885 ..."
1,16,OYAM,ABOK,ABOK,"POLYGON ((449022.853 10289925.186, 449226.234 ..."
2,17,OYAM,ACABA,ACABA,"POLYGON ((439177.884 10268289.076, 439267.859 ..."
3,18,OYAM,ALEKA,ALEKA,"POLYGON ((476164.377 10286285.074, 476167.259 ..."
4,19,OYAM,ICEME,ICEME,"POLYGON ((456784.982 10281272.815, 456864.887 ..."
...,...,...,...,...,...
115,210,OTUKE,OKWANG TOWN COUNCIL,OKWANG TOWN COUNCIL,"POLYGON ((519753.427 10285762.26, 519732.26 10..."
116,222,OTUKE,ADWARI TOWN COUNCIL,ADWARI TOWN COUNCIL,"POLYGON ((530494.768 10265816.331, 530395.634 ..."
117,19,OYAM,ICHEME TOWN COUNCIL,ICHEME TOWN COUNCIL,"POLYGON ((461430.482 10270137.159, 461485.305 ..."
118,21,OYAM,LORO TOWN COUNCIL,LORO TOWN COUNCIL,"POLYGON ((450755.966 10242606.7, 450679.183 10..."


In [37]:
# Iterate through all shapefiles in a folder and print the names of any shapefiles
# that have a field with any records containing "V", "SC", "D", "P", or "C" followed by a hyphen
def print_matching_shapefiles(folder_path):
    print(f"Searching folder: {folder_path}")
    shapefiles = list_shapefiles(folder_path)
    print(f"Shapefiles found: {shapefiles}")
    pattern = r'\b(?:V|SC|D|P|C)-'
    found = False

    if not shapefiles:
        print(f"No shapefiles found in folder: {folder_path}")
        return

    for shapefile in shapefiles:
        print(f"Reading shapefile: {shapefile}")
        gdf = read_shapefile(os.path.join(folder_path, shapefile))
        if gdf is None or gdf.empty:
            print(f"Shapefile {shapefile} is empty or could not be read.")
            continue
        for col in gdf.columns:
            if pd.api.types.is_string_dtype(gdf[col]):  # Ensure it's a Series
                if gdf[col].astype(str).str.contains(pattern, regex=True, na=False).any():
                    print(f"Pattern matched in column: {col}")
                    print(f"Matching shapefile: {shapefile}")
                    found = True
                    break

    if not found:
        print(f"Search completed: no matching shapefiles found in {folder_path}")

    print("\n")

print_matching_shapefiles("2021 subcounties")

Searching folder: 2021 subcounties
Shapefiles found: ['Lango Updated Subcounties/Lango Updated.shp', 'Lira District/Lira District.shp', 'Kole/Kole Subcounties.shp', 'Amolatar/Amolatar.shp', 'Oyam/Oyam Subcounties.shp', 'Otuke/Otuke.shp', 'Apac/Apac Subcounties.shp', 'Kwania/Kwania Subcouties.shp', 'Dokolo/Dokolo.shp', 'Lira City/Lira City.shp', 'Alebtong/Alebtong.shp']
Reading shapefile: Lango Updated Subcounties/Lango Updated.shp
Reading shapefile: Lira District/Lira District.shp
Reading shapefile: Kole/Kole Subcounties.shp
Reading shapefile: Amolatar/Amolatar.shp
Reading shapefile: Oyam/Oyam Subcounties.shp
Reading shapefile: Otuke/Otuke.shp
Reading shapefile: Apac/Apac Subcounties.shp
Reading shapefile: Kwania/Kwania Subcouties.shp
Reading shapefile: Dokolo/Dokolo.shp
Reading shapefile: Lira City/Lira City.shp
Reading shapefile: Alebtong/Alebtong.shp
Search completed: no matching shapefiles found in 2021 subcounties




In [39]:
# For all folders in unsynced_data_path, run print_matching_shapefiles function
def read_all_folders():
    for folder_name in sorted(os.listdir(unsynced_data_path)):
        folder_path = os.path.join(unsynced_data_path, folder_name)
        if os.path.isdir(folder_path):
            print_matching_shapefiles(folder_path)

read_all_folders()

Searching folder: ../../unsynced-data/Uganda_ag_ministry_shapefiles/2021 subcounties
Shapefiles found: ['Lango Updated Subcounties/Lango Updated.shp', 'Lira District/Lira District.shp', 'Kole/Kole Subcounties.shp', 'Amolatar/Amolatar.shp', 'Oyam/Oyam Subcounties.shp', 'Otuke/Otuke.shp', 'Apac/Apac Subcounties.shp', 'Kwania/Kwania Subcouties.shp', 'Dokolo/Dokolo.shp', 'Lira City/Lira City.shp', 'Alebtong/Alebtong.shp']
Reading shapefile: Lango Updated Subcounties/Lango Updated.shp
Reading shapefile: Lira District/Lira District.shp
Reading shapefile: Kole/Kole Subcounties.shp
Reading shapefile: Amolatar/Amolatar.shp
Reading shapefile: Oyam/Oyam Subcounties.shp
Reading shapefile: Otuke/Otuke.shp
Reading shapefile: Apac/Apac Subcounties.shp
Reading shapefile: Kwania/Kwania Subcouties.shp
Reading shapefile: Dokolo/Dokolo.shp
Reading shapefile: Lira City/Lira City.shp
Reading shapefile: Alebtong/Alebtong.shp
Search completed: no matching shapefiles found in ../../unsynced-data/Uganda_ag_mini

/opt/miniconda3/envs/bridges/lib/python3.13/site-packages/pyogrio/raw.py:198: RuntimeWarning: ../../unsynced-data/Uganda_ag_ministry_shapefiles/../../unsynced-data/Uganda_ag_ministry_shapefiles/UgaGIS_data/Uga_Parishes_2016/parishes_2016 - Copy.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(
/opt/miniconda3/envs/bridges/lib/python3.13/site-packages/pyogrio/raw.py:198: RuntimeWarning: ../../unsynced-data/Uganda_ag_ministry_shapefiles/../../unsynced-data/Uganda_ag_ministry_shapefiles/UgaGIS_data/Uga_Parishes_2016/parishes_2016.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


Reading shapefile: Uganda_Education.shp
Search completed: no matching shapefiles found in ../../unsynced-data/Uganda_ag_ministry_shapefiles/Uganda Education


Searching folder: ../../unsynced-data/Uganda_ag_ministry_shapefiles/Uganda Rivers
Shapefiles found: ['Uganda_Rivers.shp', 'Uganda_Rivers - Copy.shp']
Reading shapefile: Uganda_Rivers.shp
Reading shapefile: Uganda_Rivers - Copy.shp
Search completed: no matching shapefiles found in ../../unsynced-data/Uganda_ag_ministry_shapefiles/Uganda Rivers


Searching folder: ../../unsynced-data/Uganda_ag_ministry_shapefiles/Uganda Roads
Shapefiles found: ['Uganda_Roads.shp', 'Uganda_Roads - Copy.shp']
Reading shapefile: Uganda_Roads.shp
Error reading ../../unsynced-data/Uganda_ag_ministry_shapefiles/../../unsynced-data/Uganda_ag_ministry_shapefiles/Uganda Roads/Uganda_Roads.shp: IllegalArgumentException: point array must contain 0 or >1 elements

Shapefile Uganda_Roads.shp is empty or could not be read.
Reading shapefile: Uganda_Roads - Copy.

/opt/miniconda3/envs/bridges/lib/python3.13/site-packages/pyogrio/raw.py:198: RuntimeWarning: ../../unsynced-data/Uganda_ag_ministry_shapefiles/../../unsynced-data/Uganda_ag_ministry_shapefiles/Uganda Shapefile/Uganda_Shapefile.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


Search completed: no matching shapefiles found in ../../unsynced-data/Uganda_ag_ministry_shapefiles/Uganda health_facilities


Searching folder: ../../unsynced-data/Uganda_ag_ministry_shapefiles/Uganda villages
Shapefiles found: ['Uganda villages/Villages - Copy.shp', 'Uganda villages/Villages.shp']
Reading shapefile: Uganda villages/Villages - Copy.shp
Reading shapefile: Uganda villages/Villages.shp
Search completed: no matching shapefiles found in ../../unsynced-data/Uganda_ag_ministry_shapefiles/Uganda villages


Searching folder: ../../unsynced-data/Uganda_ag_ministry_shapefiles/Uganda-Subcounties-2021_Duncan
Shapefiles found: ['Uganda-Subcounties-2021.shp']
Reading shapefile: Uganda-Subcounties-2021.shp
Search completed: no matching shapefiles found in ../../unsynced-data/Uganda_ag_ministry_shapefiles/Uganda-Subcounties-2021_Duncan


Searching folder: ../../unsynced-data/Uganda_ag_ministry_shapefiles/Uganda_parishes
Shapefiles found: ['uganda_parishes_cleaned_attached/uganda_paris